# Incertitude hybride

Ce notebook teste une méthode de rejet `uncertain` qui combine deux signaux :

- la confiance sigmoid du modèle ;
- la distance aux prototypes `fresh` et `rotten` dans l'espace de features du CNN.

L'objectif est de vérifier si la combinaison des deux signaux isole mieux les prédictions risquées que chaque méthode utilisée seule.

## Principe

Une prédiction est acceptée seulement si :

```text
confidence_score >= confidence_threshold
distance_to_predicted_class_prototype <= distance_threshold
```

Sinon, elle devient `uncertain`.

Les deux seuils sont calibrés sur le `validation_set`, puis évalués sur le `test_set`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

reports_dir = project_root / "reports"
project_root

## Lancer l'analyse

Cette cellule utilise les modèles et les splits déjà sauvegardés.
Elle peut prendre plusieurs minutes car elle extrait les features de train, validation et test.

In [ ]:
command = [sys.executable, str(project_root / "src" / "hybrid_uncertainty.py"), "--protocol", "both"]
subprocess.run(command, cwd=project_root, check=True)

## Résultats globaux

In [ ]:
hybrid_metrics = pd.read_csv(reports_dir / "hybrid_uncertainty_metrics.csv")
hybrid_metrics

In [ ]:
plot_columns = ["baseline_accuracy", "accepted_accuracy", "coverage", "uncertain_rate"]
ax = hybrid_metrics.set_index("protocol")[plot_columns].plot(kind="bar", figsize=(9, 5))
ax.set_ylim(0, 1)
ax.set_title("Incertitude hybride")
ax.set_ylabel("Valeur")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Comparaison des méthodes

Cette cellule compare l'incertitude sigmoid, la distance de features et la méthode hybride quand les fichiers existent.

In [ ]:
frames = []

method_files = {
    "sigmoid_confidence": "uncertainty_metrics.csv",
    "feature_distance": "feature_distance_metrics.csv",
    "hybrid": "hybrid_uncertainty_metrics.csv",
}

for method, filename in method_files.items():
    path = reports_dir / filename
    if path.exists():
        method_df = pd.read_csv(path)
        method_df.insert(0, "method", method)
        frames.append(method_df)

if frames:
    comparison = pd.concat(frames, ignore_index=True, sort=False)
    display(comparison[["method", "protocol", "baseline_accuracy", "accepted_accuracy", "coverage", "uncertain_rate", "uncertain_error_rate"]])
else:
    print("Aucun fichier de métriques n'a été trouvé.")

## Résultats par product_type

In [ ]:
hybrid_by_product_type = pd.read_csv(reports_dir / "hybrid_uncertainty_by_product_type.csv")
hybrid_by_product_type

In [ ]:
unseen_hybrid = hybrid_by_product_type[hybrid_by_product_type["protocol"] == "unseen"]
ax = unseen_hybrid.set_index("product_type")[["baseline_accuracy", "accepted_accuracy", "uncertain_rate"]].plot(
    kind="bar",
    figsize=(9, 5),
)
ax.set_ylim(0, 1)
ax.set_title("Méthode hybride sur les catégories non vues")
ax.set_ylabel("Valeur")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Conclusion à compléter

Points à discuter après exécution :

- la méthode hybride garde-t-elle plus d'images que la distance seule ?
- atteint-elle une meilleure accuracy que le seuil sigmoid sur les catégories non vues ?
- les images rejetées contiennent-elles plus d'erreurs que les images acceptées ?
- le compromis accuracy / coverage est-il utile pour la plateforme web ?

Le but n'est pas forcément de faire mieux partout. Le but est d'évaluer une proposition construite à partir de la limite observée.